# 30 — Validate COSMoS Graph

QA pass over the Step 2 pipeline outputs. Does NOT produce a consumable artifact — emits a markdown report and a JSON sidecar for optional CI consumption.

**Inputs**
- `interim/COSMoS_Graph.xlsx` — output of notebook 10
- `interim/COSMoS_Graph_CT.xlsx` — output of notebook 20
- `cosmos-graph/reference/cosmos_linkml/cosmos_sdtm_model.yaml` — LinkML schema (for enum validation)

**Outputs**
- `reports/graph_validation_report.md` — human-readable report
- `reports/graph_validation_report.json` — machine-readable sidecar

**Checks**
1. Enumeration validation — Variables' enum-typed columns conform to schema `permissible_values`
2. Referential integrity — DSS ↔ Variables, BC ↔ DSS
3. Schema column coverage — Variables columns all representable as SDTMVariable slots
4. `vlm_source` dot-vs-hyphen detail
5. Carry-forward summary of anomalies from notebooks 10 and 20


## 1. Imports


In [ ]:
import json
import sys
from pathlib import Path
from datetime import date

import pandas as pd
from linkml_runtime.utils.schemaview import SchemaView

print(f'python : {sys.version.split()[0]}')
print(f'pandas : {pd.__version__}')


## 2. Configuration


In [ ]:
REPO_ROOT = Path.cwd().parent.parent

INTERIM = REPO_ROOT / 'cosmos-graph' / 'interim'
REPORTS = REPO_ROOT / 'cosmos-graph' / 'reports'
SCHEMAS = REPO_ROOT / 'cosmos-graph' / 'reference' / 'cosmos_linkml'

GRAPH_XLSX  = INTERIM / 'COSMoS_Graph.xlsx'
CT_XLSX     = INTERIM / 'COSMoS_Graph_CT.xlsx'
SDTM_SCHEMA = SCHEMAS / 'cosmos_sdtm_model.yaml'

REPORT_MD   = REPORTS / 'graph_validation_report.md'
REPORT_JSON = REPORTS / 'graph_validation_report.json'

for p in (GRAPH_XLSX, CT_XLSX, SDTM_SCHEMA):
    assert p.exists(), f'missing: {p}'
REPORTS.mkdir(parents=True, exist_ok=True)

print(f'graph : {GRAPH_XLSX.relative_to(REPO_ROOT)}')
print(f'ct    : {CT_XLSX.relative_to(REPO_ROOT)}')
print(f'out   : {REPORT_MD.relative_to(REPO_ROOT)}')


## 3. Load inputs


In [ ]:
dss          = pd.read_excel(GRAPH_XLSX, sheet_name='DSS')
variables    = pd.read_excel(GRAPH_XLSX, sheet_name='Variables')
relationships = pd.read_excel(GRAPH_XLSX, sheet_name='Relationships')
codelists    = pd.read_excel(GRAPH_XLSX, sheet_name='Codelists')
bc           = pd.read_excel(GRAPH_XLSX, sheet_name='BC')

ct_unresolved = pd.read_excel(CT_XLSX, sheet_name='Unresolved')
ct_anomalies  = pd.read_excel(CT_XLSX, sheet_name='Anomalies')

sv = SchemaView(str(SDTM_SCHEMA))

print(f'DSS          : {dss.shape}')
print(f'Variables    : {variables.shape}')
print(f'Relationships: {relationships.shape}')
print(f'Codelists    : {codelists.shape}')
print(f'BC           : {bc.shape}')
print()
print(f'CT Unresolved : {len(ct_unresolved)}')
print(f'CT Anomalies  : {len(ct_anomalies)}')

# Results registry — each check appends a dict
results = []


## 4. Check — enumeration validation

For each SDTMVariable slot whose range is a LinkML enum, verify that every non-null value in the corresponding Variables column is in the enum's `permissible_values`.


In [ ]:
sdtm_var_slots = sv.class_induced_slots('SDTMVariable')

# Map LinkML slot name (camelCase in schema) → Variables column name (snake_case in xlsx)
def to_snake(name):
    out = []
    for i, ch in enumerate(name):
        if ch.isupper() and i > 0:
            out.append('_')
        out.append(ch.lower())
    return ''.join(out)

enum_names = set(sv.all_enums().keys())
enum_slots = [(s.name, s.range) for s in sdtm_var_slots if s.range in enum_names]

print(f'enum-typed slots on SDTMVariable: {len(enum_slots)}')
for slot_name, enum_name in enum_slots:
    col = to_snake(slot_name)
    pv = set(sv.get_enum(enum_name).permissible_values.keys())
    print(f'  {slot_name:22s} (col={col:22s}) enum={enum_name:20s} values={len(pv)}')

enum_violations = []
for slot_name, enum_name in enum_slots:
    col = to_snake(slot_name)
    if col not in variables.columns:
        enum_violations.append({
            'slot': slot_name, 'column': col, 'enum': enum_name,
            'issue': 'column_missing', 'value': None, 'count': 0,
        })
        continue
    pv = set(sv.get_enum(enum_name).permissible_values.keys())
    actual = variables[col].dropna()
    invalid = actual[~actual.isin(pv)]
    if len(invalid):
        for val, cnt in invalid.value_counts().items():
            enum_violations.append({
                'slot': slot_name, 'column': col, 'enum': enum_name,
                'issue': 'value_not_in_enum', 'value': val, 'count': int(cnt),
            })

enum_violations_df = pd.DataFrame(enum_violations)
status = 'PASS' if enum_violations_df.empty else 'FAIL'
print(f'\nenum validation: {status} ({len(enum_violations_df)} violation rows)')
if not enum_violations_df.empty:
    print(enum_violations_df.to_string())

results.append({
    'check':   'enumeration_validation',
    'status':  status,
    'count':   len(enum_violations_df),
    'detail':  enum_violations_df.to_dict(orient='records'),
})


## 5. Check — referential integrity

Every `ds_id` in Variables must exist in DSS (and vice versa). Every `bc_id` in DSS must exist in BC.


In [ ]:
ds_ids_in_dss       = set(dss['ds_id'].dropna().unique())
ds_ids_in_variables = set(variables['ds_id'].dropna().unique())
bc_ids_in_dss       = set(dss['bc_id'].dropna().unique())
bc_ids_in_bc        = set(bc['bc_id'].dropna().unique())

integrity_issues = []
orphan_var_ds = ds_ids_in_variables - ds_ids_in_dss
orphan_dss    = ds_ids_in_dss - ds_ids_in_variables
orphan_dss_bc = bc_ids_in_dss - bc_ids_in_bc

if orphan_var_ds:
    integrity_issues.append({
        'issue': 'variables_ds_id_not_in_dss_sheet', 'count': len(orphan_var_ds),
        'examples': sorted(orphan_var_ds)[:10],
    })
if orphan_dss:
    integrity_issues.append({
        'issue': 'dss_without_any_variable_row', 'count': len(orphan_dss),
        'examples': sorted(orphan_dss)[:10],
    })
if orphan_dss_bc:
    integrity_issues.append({
        'issue': 'dss_bc_id_not_in_bc_sheet', 'count': len(orphan_dss_bc),
        'examples': sorted(orphan_dss_bc)[:10],
    })

integrity_df = pd.DataFrame(integrity_issues)
status = 'PASS' if integrity_df.empty else 'FAIL'
print(f'referential integrity: {status} ({len(integrity_df)} issue types)')
if not integrity_df.empty:
    print(integrity_df.to_string())

results.append({
    'check':   'referential_integrity',
    'status':  status,
    'count':   len(integrity_df),
    'detail':  integrity_df.to_dict(orient='records'),
})


## 6. Check — schema column coverage

Every column in `Variables` should correspond to a SDTMVariable slot (in snake_case form), with allowed exceptions for denormalised join keys (`ds_id`, `bc_id`) and the reification quad columns (`subject`, `linking_phrase`, `predicate_term`, `object`) which come from the `RelationShip` class.


In [ ]:
sdtm_var_slot_names = {to_snake(s.name) for s in sdtm_var_slots}
rel_slots = {to_snake(s.name) for s in sv.class_induced_slots('RelationShip')}
allowed = sdtm_var_slot_names | rel_slots | {'ds_id', 'bc_id'}

unknown_cols = set(variables.columns) - allowed
schema_cols_without_xlsx = sdtm_var_slot_names - set(variables.columns)

coverage_df = pd.DataFrame([
    *[{'issue': 'xlsx_column_not_in_schema', 'name': c} for c in sorted(unknown_cols)],
    *[{'issue': 'schema_slot_not_in_xlsx', 'name': s} for s in sorted(schema_cols_without_xlsx)],
])

status = 'PASS' if coverage_df.empty else 'FAIL'
print(f'schema column coverage: {status} ({len(coverage_df)} mismatches)')
if not coverage_df.empty:
    print(coverage_df.to_string())

results.append({
    'check':   'schema_column_coverage',
    'status':  status,
    'count':   len(coverage_df),
    'detail':  coverage_df.to_dict(orient='records'),
})


## 7. Check — vlm_source dot-vs-hyphen detail (INFO)

Notebook 10 counts hyphenated `vlm_source` values. Here we list the distinct patterns and the DSSs affected, so reviewers can see whether this is a single rogue DSS or a broader authoring inconsistency.


In [ ]:
# Reload structural source column from the raw merge of domain.source — it's present in DSS.source
hyphen_mask = dss['source'].str.contains('-', na=False)
hyphen_rows = dss.loc[hyphen_mask, ['ds_id', 'domain', 'source', 'ds_short_name']].copy()

vlm_source_df = hyphen_rows.reset_index(drop=True)
print(f'DSSs with hyphen-format source: {len(vlm_source_df)}')
if len(vlm_source_df):
    print(vlm_source_df.to_string())

results.append({
    'check':   'vlm_source_hyphen_detail',
    'status':  'INFO',
    'count':   len(vlm_source_df),
    'detail':  vlm_source_df.to_dict(orient='records'),
})


## 8. Carry-forward — anomalies from notebooks 10 and 20

Consolidated here for a single-document view. Authoritative counts remain in the source notebooks.


In [ ]:
# From notebook 10 — empty reification quads (Variables rows without quad, not in Relationships)
empty_quad_count = variables.shape[0] - relationships.shape[0]

# From notebook 10 — DSSs without any relationship edge
dss_with_edges = set(relationships['ds_id'].unique())
dss_without_edges = sorted(set(dss['ds_id']) - dss_with_edges)

carry_forward = [
    {
        'check':  'empty_reification_quad_rows',
        'status': 'INFO', 'count': empty_quad_count,
        'detail': [{'note': 'Variables rows without a reification quad; not counted in Relationships sheet'}],
    },
    {
        'check':  'dss_without_any_edge',
        'status': 'INFO', 'count': len(dss_without_edges),
        'detail': [{'ds_id': d} for d in dss_without_edges],
    },
    {
        'check':  'ct_unresolved_concept_ids',
        'status': 'INFO' if len(ct_unresolved) == 0 else 'FAIL',
        'count':  len(ct_unresolved),
        'detail': ct_unresolved.to_dict(orient='records'),
    },
    {
        'check':  'pinned_term_not_in_bound_codelist',
        'status': 'INFO' if len(ct_anomalies) == 0 else 'FAIL',
        'count':  len(ct_anomalies),
        'detail': ct_anomalies.to_dict(orient='records'),
    },
]
results.extend(carry_forward)

for r in carry_forward:
    print(f'  {r["check"]:40s} : {r["status"]:5s} ({r["count"]})')


## 9. Build markdown + JSON report


In [ ]:
def _esc(v):
    if pd.isna(v) or v is None:
        return ''
    return str(v).replace('|', '\\|').replace('\n', ' ')

def _detail_table(records, max_rows=25):
    if not records:
        return '_no details_\n'
    df = pd.DataFrame(records)
    truncated = len(df) > max_rows
    view = df.head(max_rows)
    cols = list(view.columns)
    lines = []
    lines.append('| ' + ' | '.join(cols) + ' |')
    lines.append('| ' + ' | '.join(['---'] * len(cols)) + ' |')
    for _, row in view.iterrows():
        lines.append('| ' + ' | '.join(_esc(row[c]) for c in cols) + ' |')
    if truncated:
        lines.append(f'\n_showing first {max_rows} of {len(df)} — full list in JSON sidecar_')
    return '\n'.join(lines) + '\n'

lines = []
lines.append('# COSMoS Graph Validation Report\n')
lines.append(f'_Generated: {date.today().isoformat()}_\n')
lines.append(f'Inputs: `{GRAPH_XLSX.name}`, `{CT_XLSX.name}`\n')
lines.append('\n## Summary\n')
lines.append('| Check | Status | Count |')
lines.append('| --- | --- | --- |')
for r in results:
    lines.append(f'| {r["check"]} | {r["status"]} | {r["count"]} |')
lines.append('')
lines.append('\n## Details\n')
for r in results:
    lines.append(f'### {r["check"]} — {r["status"]} ({r["count"]})\n')
    lines.append(_detail_table(r['detail']))
    lines.append('')

report_md = '\n'.join(lines)

REPORT_MD.write_text(report_md, encoding='utf-8')
print(f'wrote {REPORT_MD} ({REPORT_MD.stat().st_size:,} bytes)')

report_json = {
    'generated':     date.today().isoformat(),
    'inputs':        [GRAPH_XLSX.name, CT_XLSX.name, SDTM_SCHEMA.name],
    'pipeline_note': 'generated by cosmos-graph/notebooks/30_validate_graph.ipynb',
    'results':       results,
}
REPORT_JSON.write_text(json.dumps(report_json, indent=2, default=str), encoding='utf-8')
print(f'wrote {REPORT_JSON} ({REPORT_JSON.stat().st_size:,} bytes)')


## 10. Echo report summary


In [ ]:
print(f'{"Check":40s} {"Status":6s} {"Count":>6s}')
print('-' * 60)
for r in results:
    print(f'{r["check"]:40s} {r["status"]:6s} {r["count"]:>6}')

fail_count = sum(1 for r in results if r['status'] == 'FAIL')
print(f'\n{fail_count} check(s) with status FAIL')
